# Final Model — Hierarchical NIDS with Tempered Class Weighting

**CIC-UNSW-NB15-2024 · 10-class network intrusion detection**

The consolidated pipeline. The working record lives across the experiment notebooks
(`two_stage_*`, `label_ambiguity_ceiling`, `knn_class_overlap`, `ablation_significance`); this is the
single end-to-end pass, following the project's seven stages.

## The model in one paragraph

The 10-class problem is **restructured** into a binary gate (Benign vs Attack) followed by a 9-class
attack-family classifier. The family classifier is an **ensemble of seven XGBoost-class models
deliberately spread along the class-weighting axis**, combined by a **per-class weight matrix**
$W \in \mathbb{R}^{7\times 9}$ fitted by coordinate ascent on macro-F1. Class weights follow
**tempered inverse frequency** $w_c \propto (1/n_c)^{0.25}$ — not `balanced`, which measurably hurts.
All probabilities feeding the combiner are **cross-fitted out-of-fold**, so no model scores its own
training rows.

## Headline result

| | macro-F1 | 95% CI |
|---|---|---|
| Flat XGBoost baseline (`balanced`) | 0.580 | [0.564, 0.596] |
| **This model** | **≈0.630** | **[0.611, 0.649]** |
| Improvement (paired) | **+0.050** | [0.039, 0.061], p < 0.001 |

**Macro-F1 is the primary metric** — accuracy is Benign-dominated (73.5% of the test set) and
misleading. Every figure is reported with a bootstrap confidence interval; the test set resolves
differences of roughly ±0.012 (paired), so fourth decimals are not meaningful.

## Two framing points carried from the analysis

1. **The hierarchy and the weighting are one mechanism, not two.** The two-stage structure is
   *negative* under `balanced` weighting (−0.006, p = 0.174) and significantly positive under α=0.25
   (+0.018, p = 0.005). It should never be claimed as a standalone improvement.
2. **The ~0.63 plateau is a property of the feature representation.** k-NN overlap analysis puts the
   Bayes error of the family sub-problem at **21–38%** on these 44 features, and this model already
   sits near the favourable end of that bracket. Remaining error is not a modelling deficiency.

> **Runtime ≈ 12 min**, dominated by the 5-fold cross-fitting in Stage 4.

In [1]:
%pip install xgboost imbalanced-learn -q

import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score, roc_auc_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

RS = 42
LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
NAMES = [LABELS[i] for i in range(10)]
print("ready")

Note: you may need to restart the kernel to use updated packages.
ready


# 1. Data Collection — *Sources & Quality*

### 1.1 Load and confirm row alignment

Features and labels are in two files with **no shared key** — they are aligned by position only.
Nothing may reorder one without the other.

In [2]:
X_raw = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y_raw = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]

assert len(X_raw) == len(y_raw), "row counts differ - files are not aligned"
print("features : %s" % (X_raw.shape,))
print("labels   : %s" % (y_raw.shape,))
print("aligned by position, %d rows" % len(X_raw))
print("\nlabel values present:", sorted(y_raw.unique()))
print("mapping (alphabetical attack families, Benign = 0):")
print("  " + " | ".join("%d %s" % (k, v) for k, v in LABELS.items()))

features : (447915, 76)
labels   : (447915,)
aligned by position, 447915 rows

label values present: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
mapping (alphabetical attack families, Benign = 0):
  0 Benign | 1 Analysis | 2 Backdoor | 3 DoS | 4 Exploits | 5 Fuzzers | 6 Generic | 7 Reconnaissance | 8 Shellcode | 9 Worms


### 1.2 Quality audit

Establishing what the data does *not* need before deciding what it does.

In [3]:
print("NaNs               : %d" % X_raw.isna().sum().sum())
print("infinities         : %d" % np.isinf(X_raw.select_dtypes(include=[np.number])).sum().sum())
print("non-numeric columns: %d" % (X_raw.dtypes == object).sum())
print("dtypes             : %s" % dict(X_raw.dtypes.value_counts().items()))
print("\n-> no imputation or encoding required")

dist = y_raw.value_counts().sort_index()
print("\nraw class distribution:")
for c, n in dist.items():
    print("  %-15s %7d  %6.3f%%" % (LABELS[c], n, 100 * n / len(y_raw)))
print("\nimbalance ratio (Benign : Worms) = %.0f : 1" % (dist[0] / dist[9]))

NaNs               : 0
infinities         : 0
non-numeric columns: 0
dtypes             : {dtype('float64'): 45, dtype('int64'): 31}

-> no imputation or encoding required

raw class distribution:
  Benign           358332  80.000%
  Analysis            385   0.086%
  Backdoor            452   0.101%
  DoS                4467   0.997%
  Exploits          30951   6.910%
  Fuzzers           29613   6.611%
  Generic            4632   1.034%
  Reconnaissance    16735   3.736%
  Shellcode          2102   0.469%
  Worms               246   0.055%

imbalance ratio (Benign : Worms) = 1457 : 1


# 2. Data Preparation — *Clean & Transform*

The bulk of the work, and the stage where correctness is won or lost. Every decision below is made
**after** showing the evidence for it, never assumed.

### 2.1 Duplicate check

The single largest correctness trap in this dataset. Duplicates that straddle a train/test split leak
test rows into training and produce near-perfect scores that mean nothing — the same artefact the
literature review flagged in three of the reviewed papers.

In [4]:
full = pd.concat([X_raw, y_raw], axis=1)
n_dup = full.duplicated().sum()
print("exact duplicate rows (features + label): %d  (%.2f%%)" % (n_dup, 100 * n_dup / len(full)))
print("\nif left in place, a stratified 80/20 split would place copies of the same flow")
print("on both sides of the split -> inflated, meaningless scores.")

exact duplicate rows (features + label): 141742  (31.64%)

if left in place, a stratified 80/20 split would place copies of the same flow
on both sides of the split -> inflated, meaningless scores.


### 2.2 Drop duplicates

Deduplication is on **features + label**, so a feature vector carrying two different labels is kept
twice — genuine ambiguity is preserved rather than silently resolved.

In [5]:
df = full.drop_duplicates().reset_index(drop=True)
X, y = df[X_raw.columns], df["Label"]
print("%d -> %d rows (%d removed)" % (len(full), len(df), len(full) - len(df)))

447915 -> 306173 rows (141742 removed)


### 2.3 Class distribution after deduplication

The imbalance that drives every subsequent design decision.

In [6]:
d = y.value_counts().sort_index()
print(pd.DataFrame({"count": d, "pct": (100 * d / len(y)).round(3)},
                   index=[LABELS[i] for i in d.index]).to_string())
print("\nBenign share %.1f%% | rarest class %s at %d rows"
      % (100 * d[0] / len(y), LABELS[d.idxmin()], d.min()))
print("-> plain accuracy is uninformative; macro-F1 is the primary metric")

                count  pct
Benign            NaN  NaN
Analysis          NaN  NaN
Backdoor          NaN  NaN
DoS               NaN  NaN
Exploits          NaN  NaN
Fuzzers           NaN  NaN
Generic           NaN  NaN
Reconnaissance    NaN  NaN
Shellcode         NaN  NaN
Worms             NaN  NaN

Benign share 73.5% | rarest class Worms at 241 rows
-> plain accuracy is uninformative; macro-F1 is the primary metric


### 2.4 Variance check

Surfacing constant columns. They carry no information and only add noise to downstream selection.

In [7]:
nun  = X.nunique()
zero = nun[nun <= 1].index.tolist()
print("zero-variance columns: %d" % len(zero))
for c in zero:
    print("  %-28s constant value %s" % (c, X[c].iloc[0]))

zero-variance columns: 9
  Bwd PSH Flags                constant value 0
  Fwd URG Flags                constant value 0
  Bwd URG Flags                constant value 0
  URG Flag Count               constant value 0
  CWR Flag Count               constant value 0
  ECE Flag Count               constant value 0
  Fwd Bytes/Bulk Avg           constant value 0
  Fwd Packet/Bulk Avg          constant value 0
  Fwd Bulk Rate Avg            constant value 0


### 2.5 Correlation check

Surfacing near-duplicate columns at |r| ≥ 0.95. This prune is **label-free**, so it may legitimately
precede the split — no information about `y` is used.

In [8]:
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
print("redundant columns at |r| >= 0.95: %d" % len(red))
for c in red[:12]:
    partner = upper[c].idxmax()
    print("  %-28s r = %.3f with %s" % (c, upper[c].max(), partner))
if len(red) > 12:
    print("  ... and %d more" % (len(red) - 12))

redundant columns at |r| >= 0.95: 23
  Total Length of Bwd Packet   r = 0.969 with Total Bwd packets
  Fwd Packet Length Std        r = 0.954 with Fwd Packet Length Max
  Bwd Packet Length Std        r = 0.953 with Bwd Packet Length Max
  Fwd IAT Total                r = 0.999 with Flow Duration
  Fwd IAT Mean                 r = 0.982 with Flow IAT Std
  Fwd IAT Std                  r = 0.989 with Flow IAT Std
  Fwd IAT Max                  r = 0.997 with Flow IAT Max
  Fwd IAT Min                  r = 0.985 with Flow IAT Min
  Bwd IAT Std                  r = 0.985 with Bwd IAT Mean
  Fwd Header Length            r = 0.981 with Total Fwd Packet
  Bwd Header Length            r = 0.975 with Total Bwd packets
  Fwd Packets/s                r = 0.999 with Flow Packets/s
  ... and 11 more


### 2.6 Drop uninformative and redundant features

76 → 44. Independently validated later: the label-ambiguity analysis showed this prune creates **zero**
new label ambiguity, i.e. the dropped columns separated no pair of otherwise-identical flows.

In [9]:
DROP = list(dict.fromkeys(zero + red))
X = X.drop(columns=DROP)
FEATURES = list(X.columns)
print("%d -> %d features (%d dropped)" % (len(X_raw.columns), len(FEATURES), len(DROP)))
print("\ntree models are scale-invariant, so no scaling is applied to the pipeline itself.")

76 -> 44 features (32 dropped)

tree models are scale-invariant, so no scaling is applied to the pipeline itself.


### 2.7 Stratified train/test split

80/20, stratified so the rare families appear in both sides in proportion. The test set is held out
until Stage 5 and touched once.

In [10]:
X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_trf = X_trf.reset_index(drop=True); y_trf = y_trf.reset_index(drop=True)
y_arr, y_test_arr = y_trf.values, y_test.values

print("train+val %d | test %d" % (len(y_trf), len(y_test)))
print(pd.DataFrame({"train+val": y_trf.value_counts().sort_index(),
                    "test":      y_test.value_counts().sort_index()},
                   index=NAMES).to_string())

train+val 244938 | test 61235
                train+val  test
Benign                NaN   NaN
Analysis              NaN   NaN
Backdoor              NaN   NaN
DoS                   NaN   NaN
Exploits              NaN   NaN
Fuzzers               NaN   NaN
Generic               NaN   NaN
Reconnaissance        NaN   NaN
Shellcode             NaN   NaN
Worms                 NaN   NaN


# 3. Model Selection — *Algorithm Choice*

Four design decisions, each resting on a measured result rather than a preference.

### 3.1 Restructure the problem: gate + family classifier

The 10-class problem is split into **Stage A** (Benign vs Attack) and **Stage B** (9 attack families,
trained on attack rows only). Stage A reaches ~0.997 attack recall; the difficulty is concentrated
entirely in Stage B, which lets all subsequent effort be spent where it matters.

**Important caveat, established by the ablation study:** the hierarchy is **not** an improvement on its
own. Under `balanced` weighting it measures −0.006 (p = 0.174) against a flat classifier. It becomes
significantly positive (+0.018, p = 0.005) **only** in combination with the tempered weighting in 3.2.
Hierarchy and weighting are one mechanism.

### 3.2 Tempered class weighting, α = 0.25

$$w_c \;\propto\; (1/n_c)^{\alpha}, \qquad \alpha = 0.25$$

α = 1 is exactly `class_weight='balanced'`; α = 0 is no weighting. The weighting study swept the axis
and found an **interior optimum**: `balanced` ranked 12th of 14 schemes, *below no weighting at all*.
The effective-number family (Cui et al. 2019) independently peaks in the same place once expressed as
a weight ratio — the useful band is **3–11×**, where `balanced` demands 128×.

This is the project's largest single lever: **+0.0385 [0.0246, 0.0512]**, more than twice the
contribution of the seven-member ensemble.

### 3.3 An α-diverse pool

Seven members. Four are the *same* XGBoost at different α, deliberately spanning the precision↔recall
axis rather than each being individually optimal; one uses capped SMOTE for a resampling-induced bias;
RF and ExtraTrees add algorithmic variety.

The decisive lesson from the ensemble work: **members are selected for complementarity, not quality.**
An earlier pool diversified by *algorithm* (XGB/RF/ET at fixed α) failed entirely — the models agreed
too often. α = 1.0 is a poor standalone model yet supplies most of the usable diversity. Extending α
beyond 1.0 was also tested and **rejected**: α = 1.5/2.0 disagree more but are simply degraded, and
adding them lowered the score.

### 3.4 Per-class weight matrix as the combiner

$$S(c \mid x) \;=\; \sum_{m} W[m,c]\; P_m(c \mid x), \qquad W \in \mathbb{R}^{7\times 9}$$

Each family is scored by its *own* linear combination of the pool, so a recall-heavy member can carry
DoS while a precision-heavy one carries Shellcode. Stacking meta-learners were tested and **rejected** —
a depth-3 booster on the same 63 meta-features scored 0.918 in-sample and 0.593 on test. Same parameter
count, worse generalisation: the difference is inductive bias, and constrained combiners win when the
combiner-fitting set is small.

A further benefit: because fusion is linear in member probabilities, per-class SHAP attributions
compose exactly as $\sum_m W[m,c]\,\phi_m$ — the architecture stays explainable in a way a stacked
meta-learner would not.

In [11]:
ALPHA = 0.25

def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)          # let the wrapper infer the objective:
                                       # passing num_class makes the binary gate return a 2-D array

def wc_alpha(n, a):
    """tempered inverse frequency; a=0 uniform, a=1 == class_weight='balanced'."""
    w = (n.sum() / n) ** a
    return w / w.mean()

def sample_w(ya, wc):
    w = wc[ya].astype(float)
    return w * len(w) / w.sum()

# family index space: labels 1..9 -> 0..8
attack_labels = sorted(l for l in y_trf.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC  = len(attack_labels)
FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]

att_trf = y_arr != 0
fam_trf = np.full(len(y_arr), -1)
fam_trf[att_trf] = pd.Series(y_arr[att_trf]).map(fam_to_idx).values
n_att = np.bincount(fam_trf[att_trf], minlength=NC)

SPECS = [(f"XGB a={a}", ("alpha", a)) for a in (0.0, 0.25, 0.5, 1.0)] + \
        [("XGB smote2000", ("smote", 2000)),
         ("RF a=0.25",     ("rf", ALPHA)),
         ("ET a=0.25",     ("et", ALPHA))]
MODELS = [s[0] for s in SPECS]; M = len(MODELS)

def fit_member(kind, param, Xa, ya):
    n = np.bincount(np.asarray(ya), minlength=NC)
    if kind == "smote":
        strat = {c: param for c in range(NC) if n[c] < param}
        Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(Xa, ya)
        m = make_xgb(); m.fit(Xs, ys); return m
    if   kind == "alpha": m = make_xgb()
    elif kind == "rf":    m = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    elif kind == "et":    m = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS)
    else: raise ValueError(kind)
    m.fit(Xa, ya, sample_weight=sample_w(np.asarray(ya), wc_alpha(n, param)))
    return m

def fit_gate(Xa, ya10):
    yb = (ya10 != 0).astype(int)
    g = make_xgb(); g.fit(Xa, yb, sample_weight=compute_sample_weight("balanced", yb))
    return g

# what the weighting actually does
print("class weights across the pool's alpha values (Stage-B families):\n")
print(pd.DataFrame({("a=%.2f" % a): wc_alpha(n_att, a).round(3) for a in (0.0, 0.25, 0.5, 1.0)},
                   index=FAM).to_string())
print("\nWorms:Exploits weight ratio -> " + ", ".join(
    "a=%.2f: %.1fx" % (a, wc_alpha(n_att, a)[FAM.index("Worms")] / wc_alpha(n_att, a)[FAM.index("Exploits")])
    for a in (0.0, 0.25, 0.5, 1.0)))

class weights across the pool's alpha values (Stage-B families):

                a=0.00  a=0.25  a=0.50  a=1.00
Analysis           1.0   1.513   1.939   2.315
Backdoor           1.0   1.453   1.789   1.970
DoS                1.0   0.822   0.573   0.202
Exploits           1.0   0.505   0.216   0.029
Fuzzers            1.0   0.526   0.234   0.034
Generic            1.0   0.818   0.567   0.198
Reconnaissance     1.0   0.640   0.347   0.074
Shellcode          1.0   1.022   0.885   0.483
Worms              1.0   1.700   2.450   3.695

Worms:Exploits weight ratio -> a=0.00: 1.0x, a=0.25: 3.4x, a=0.50: 11.3x, a=1.00: 128.1x


# 4. Model Training — *Learning Process*

### 4.1 Why cross-fitting

The combiner $W$ must be fitted on probabilities the base models did **not** produce for their own
training rows — otherwise the members look overconfident and the weights are fitted to an illusion.

The naive fix is to hold out a slice of data for the combiner, which wastes it. **5-fold cross-fitting**
avoids the trade-off: each fold trains the pool on the other four and predicts the held-out fold,
producing honest out-of-sample probabilities for *every* row. $W$ is then fitted on all ~65,000 attack
rows, and the pool is refit on the full data for deployment.

An earlier protocol fitted $W$ on a 6,494-row holdout; this uses **5× more** and leaves nothing unused.

In [12]:
OOF = [np.zeros((len(y_arr), NC), np.float32) for _ in range(M)]

t0 = time.time()
for f, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=RS).split(X_trf, y_arr)):
    tf = time.time()
    Xtr_f, ytr_f = X_trf.iloc[tr], y_arr[tr]
    m_att = ytr_f != 0
    Xa = Xtr_f[m_att].reset_index(drop=True)
    ya = pd.Series(fam_trf[tr][m_att])
    for mi, (_, (kind, param)) in enumerate(SPECS):
        OOF[mi][va] = fit_member(kind, param, Xa, ya).predict_proba(X_trf.iloc[va]).astype(np.float32)
    print("  fold %d/5 (%.0fs)" % (f + 1, time.time() - tf))
print("cross-fitting complete in %.0fs" % (time.time() - t0))

  fold 1/5 (61s)
  fold 2/5 (58s)
  fold 3/5 (54s)
  fold 4/5 (53s)
  fold 5/5 (53s)
cross-fitting complete in 279s


### 4.2 Fit the per-class weight matrix

Coordinate ascent directly on macro-F1 over the out-of-fold probabilities. 63 parameters, a coarse
grid, three passes — deliberately constrained, because the combiner-fitting signal for the rarest
families is thin.

In [13]:
att_idx = np.where(att_trf)[0]
yf = fam_trf[att_idx]
GRID = (0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0)

def fuse(probas, W):
    S = np.zeros(probas[0].shape, np.float64)
    for i in range(len(probas)):
        S += probas[i] * W[i][None, :]
    return S

t0 = time.time()
P_oof = [OOF[m][att_idx] for m in range(M)]
W = np.ones((M, NC)) / M
best = f1_score(yf, fuse(P_oof, W).argmax(1), average="macro", zero_division=0)
for _ in range(3):
    for c in range(NC):
        for i in range(M):
            cur = W[i, c]
            for g in GRID:
                W[i, c] = g
                if W[:, c].sum() == 0:
                    continue
                s = f1_score(yf, fuse(P_oof, W).argmax(1), average="macro", zero_division=0)
                if s > best + 1e-7:
                    best, cur = s, g
            W[i, c] = cur
print("W fitted on %d OOF attack rows in %.0fs -> Stage-B macro-F1 %.4f"
      % (len(yf), time.time() - t0, best))

Wn = W / np.maximum(W.sum(0, keepdims=True), 1e-9)
print("\nnormalised weight matrix (column = family):")
print(pd.DataFrame(Wn, index=MODELS, columns=FAM).round(3).to_string())
print("\ndominant member per family:")
for c, fm in enumerate(FAM):
    print("  %-15s %-14s %.2f" % (fm, MODELS[Wn[:, c].argmax()], Wn[:, c].max()))

W fitted on 64936 OOF attack rows in 27s -> Stage-B macro-F1 0.6209

normalised weight matrix (column = family):
               Analysis  Backdoor    DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
XGB a=0.0         0.000     0.000  0.568     0.000    0.109    0.000           0.149      0.515  0.000
XGB a=0.25        0.090     0.000  0.108     0.000    0.062    0.154           0.000      0.059  0.154
XGB a=0.5         0.538     0.211  0.000     0.211    0.328    0.269           0.085      0.206  0.269
XGB a=1.0         0.179     0.211  0.108     0.000    0.438    0.269           0.000      0.059  0.154
XGB smote2000     0.090     0.211  0.000     0.368    0.062    0.154           0.596      0.103  0.154
RF a=0.25         0.051     0.000  0.108     0.211    0.000    0.000           0.085      0.059  0.269
ET a=0.25         0.051     0.368  0.108     0.211    0.000    0.154           0.085      0.000  0.000

dominant member per family:
  Analysis        XGB a=0.5      0

### 4.3 Refit on the full training data

Gate and pool are refit on all of `train+val`. $W$ stays as fitted in 4.2 — this is the standard
cross-fitting arrangement, and it biases slightly *against* the model (the deployed members are
marginally sharper than the ones $W$ was tuned against).

In [14]:
t0 = time.time()
GATE = fit_gate(X_trf, y_arr)
Xa_f = X_trf[att_trf].reset_index(drop=True)
ya_f = pd.Series(fam_trf[att_trf])
POOL = [fit_member(k, p, Xa_f, ya_f) for _, (k, p) in SPECS]
print("gate + %d members refit on %d rows in %.0fs" % (M, len(y_arr), time.time() - t0))

gate + 7 members refit on 244938 rows in 73s


# 5. Model Evaluation — *Test Performance*

The test set is used here for the first time.

In [15]:
gate_test = GATE.predict(X_test)
p_att     = GATE.predict_proba(X_test)[:, 1]
P_test    = [m.predict_proba(X_test).astype(np.float32) for m in POOL]
S_test    = fuse(P_test, W)
fam_test  = S_test.argmax(1)

def route(fam_all, gate_pred):
    yp = np.zeros(len(gate_pred), int)
    g = gate_pred == 1
    yp[g] = np.vectorize(idx_to_fam.get)(fam_all[g])
    return yp

y_pred = route(fam_test, gate_test)

def macro_f1_fast(yt, yp, C=10):
    cm = np.bincount(yt * C + yp, minlength=C * C).reshape(C, C)
    tp = np.diag(cm).astype(float)
    den = 2 * tp + (cm.sum(0) - tp) + (cm.sum(1) - tp)
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1e-12), 0.0)
    return f1.mean(), f1

# stratified bootstrap: resample within class, matching the stratified split design
B = 2000
rng = np.random.default_rng(RS)
cls_idx = [np.where(y_test_arr == c)[0] for c in range(10)]
BOOT_IDX = [np.concatenate([rng.choice(ci, size=len(ci), replace=True) for ci in cls_idx])
            for _ in range(B)]

def boot_ci(pred, stat="macro"):
    vals = np.array([macro_f1_fast(y_test_arr[i], pred[i])[0] for i in BOOT_IDX])
    return np.percentile(vals, [2.5, 97.5]), vals

(ci_lo, ci_hi), boot_final = boot_ci(y_pred)
print("=== HEADLINE (test, %d rows) ===" % len(y_test_arr))
print("macro-F1  : %.4f   95%% CI [%.4f, %.4f]" % (macro_f1_fast(y_test_arr, y_pred)[0], ci_lo, ci_hi))
print("macro-P   : %.4f" % precision_score(y_test_arr, y_pred, average="macro", zero_division=0))
print("macro-R   : %.4f" % recall_score(y_test_arr, y_pred, average="macro", zero_division=0))
print("accuracy  : %.4f  (Benign-dominated -- not the primary metric)"
      % (y_pred == y_test_arr).mean())
print("\nweighted-F1: %.4f" % f1_score(y_test_arr, y_pred, average="weighted", zero_division=0))
print("\ngate: attack recall %.4f / precision %.4f"
      % (recall_score((y_test_arr != 0).astype(int), gate_test),
         precision_score((y_test_arr != 0).astype(int), gate_test)))

=== HEADLINE (test, 61235 rows) ===
macro-F1  : 0.6336   95% CI [0.6145, 0.6517]
macro-P   : 0.6764
macro-R   : 0.6472
accuracy  : 0.9055  (Benign-dominated -- not the primary metric)

weighted-F1: 0.9089

gate: attack recall 0.9966 / precision 0.9175


### 5.2 Per-class performance with confidence intervals

The rare families carry wide intervals — Worms has 48 test rows. **Report them as intervals**; a point
estimate to three decimals is not defensible at that support, and comparisons inside the interval
cannot be claimed.

In [16]:
pc_boot = np.array([macro_f1_fast(y_test_arr[i], y_pred[i])[1] for i in BOOT_IDX])
pt_pc   = macro_f1_fast(y_test_arr, y_pred)[1]
rep     = classification_report(y_test_arr, y_pred, labels=list(range(10)),
                                target_names=NAMES, output_dict=True, zero_division=0)
rows = []
for c in range(10):
    lo, hi = np.percentile(pc_boot[:, c], [2.5, 97.5])
    rows.append(dict(cls=NAMES[c], support=int(rep[NAMES[c]]["support"]),
                     precision=round(rep[NAMES[c]]["precision"], 3),
                     recall=round(rep[NAMES[c]]["recall"], 3),
                     F1=round(pt_pc[c], 3),
                     ci="[%.3f, %.3f]" % (lo, hi), width=round(hi - lo, 3)))
print(pd.DataFrame(rows).to_string(index=False))

           cls  support  precision  recall    F1             ci  width
        Benign    45001      0.999   0.968 0.983 [0.982, 0.984]  0.002
      Analysis       77      0.312   0.818 0.452 [0.400, 0.506]  0.106
      Backdoor       90      0.828   0.533 0.649 [0.556, 0.732]  0.176
           DoS      883      0.618   0.375 0.467 [0.435, 0.497]  0.062
      Exploits     6182      0.892   0.673 0.767 [0.758, 0.775]  0.017
       Fuzzers     5279      0.572   0.934 0.710 [0.703, 0.716]  0.013
       Generic      899      0.841   0.789 0.814 [0.794, 0.834]  0.040
Reconnaissance     2406      0.655   0.625 0.640 [0.624, 0.655]  0.031
     Shellcode      370      0.356   0.384 0.369 [0.327, 0.412]  0.085
         Worms       48      0.692   0.375 0.486 [0.347, 0.623]  0.277


### 5.3 Confusion matrix

The structure here is not arbitrary: the k-NN overlap analysis showed **Exploits and Fuzzers act as
attractors** in feature space, with the smaller families geometrically embedded inside them (Analysis
is 51% Exploits by neighbourhood, Shellcode 47% Fuzzers). The off-diagonal mass follows that geometry —
overlap predicts confusion at ρ = 0.83.

In [17]:
cm = pd.DataFrame(confusion_matrix(y_test_arr, y_pred, labels=list(range(10))),
                  index=NAMES, columns=NAMES)
print(cm.to_string())
print("\nlargest off-diagonal cells:")
cmv = cm.values.copy(); np.fill_diagonal(cmv, 0)
for _ in range(6):
    a, b = np.unravel_index(cmv.argmax(), cmv.shape)
    print("  %-15s -> %-15s %5d" % (NAMES[a], NAMES[b], cmv[a, b]))
    cmv[a, b] = 0

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign           43546        52         3   10        59     1174       13             110         34      0
Analysis             1        63         0    5         8        0        0               0          0      0
Backdoor             0         0        48    0        13       19        0               9          0      1
DoS                  2        19         1  331       210      230       14              53         23      0
Exploits             9        66         1  159      4158     1197       58             471         59      4
Fuzzers             32         2         2    9        47     4929       43             100        112      3
Generic              2         0         1    9        27      123      709              18         10      0
Reconnaissance       9         0         1    9       117      748        1            1503         18      0
Shellcode 

### 5.4 ROC-AUC (one-vs-rest, macro)

Requires calibrated 10-class probabilities, which hard routing does not provide. They are composed
from the two stages:

$$P(\text{Benign}) = 1 - p_{\text{att}}, \qquad P(k) = p_{\text{att}} \cdot P_B(k)$$

This composition is used **only** for the ranking metrics; the reported predictions above come from
hard routing, which is what the ablation and all comparisons evaluated.

In [18]:
PB = S_test / np.maximum(S_test.sum(1, keepdims=True), 1e-12)
P10 = np.zeros((len(y_test_arr), 10))
P10[:, 0] = 1.0 - p_att
for i in range(NC):
    P10[:, idx_to_fam[i]] = p_att * PB[:, i]
P10 /= P10.sum(1, keepdims=True)

print("ROC-AUC (OvR, macro)   : %.4f" % roc_auc_score(y_test_arr, P10, multi_class="ovr", average="macro"))
print("ROC-AUC (OvR, weighted): %.4f" % roc_auc_score(y_test_arr, P10, multi_class="ovr", average="weighted"))
print("\nper-class OvR AUC:")
for c in range(10):
    print("  %-15s %.4f" % (NAMES[c], roc_auc_score((y_test_arr == c).astype(int), P10[:, c])))

ROC-AUC (OvR, macro)   : 0.9818
ROC-AUC (OvR, weighted): 0.9908

per-class OvR AUC:
  Benign          0.9946
  Analysis        0.9987
  Backdoor        0.9795
  DoS             0.9608
  Exploits        0.9824
  Fuzzers         0.9798
  Generic         0.9911
  Reconnaissance  0.9811
  Shellcode       0.9736
  Worms           0.9760


### 5.5 Against the baseline — paired significance

A flat XGBoost with `balanced` weighting, the project's original baseline, trained on the same data.
The difference is bootstrapped **paired** (both models scored on the same resample each iteration),
which is the correct and much tighter comparison.

In [19]:
t0 = time.time()
base = make_xgb(); base.fit(X_trf, y_arr, sample_weight=compute_sample_weight("balanced", y_arr))
y_base = base.predict(X_test)
print("baseline trained in %.0fs\n" % (time.time() - t0))

_, boot_base = boot_ci(y_base)
d = boot_final - boot_base
lo, hi = np.percentile(d, [2.5, 97.5])
p = 2 * min((d <= 0).mean(), (d >= 0).mean())

print(pd.DataFrame([
    dict(model="flat XGBoost, balanced (baseline)",
         macro_F1=round(macro_f1_fast(y_test_arr, y_base)[0], 4),
         ci="[%.3f, %.3f]" % tuple(np.percentile(boot_base, [2.5, 97.5]))),
    dict(model="hierarchical + tempered weighting + ensemble",
         macro_F1=round(macro_f1_fast(y_test_arr, y_pred)[0], 4),
         ci="[%.3f, %.3f]" % (ci_lo, ci_hi)),
]).to_string(index=False))
print("\nimprovement: %+.4f   95%% CI [%.4f, %.4f]   p = %.4f"
      % (macro_f1_fast(y_test_arr, y_pred)[0] - macro_f1_fast(y_test_arr, y_base)[0], lo, hi, min(p, 1.0)))
print("-> %s" % ("ESTABLISHED" if lo > 0 else "not established"))

baseline trained in 35s

                                       model  macro_F1             ci
           flat XGBoost, balanced (baseline)    0.5807 [0.564, 0.596]
hierarchical + tempered weighting + ensemble    0.6336 [0.614, 0.652]

improvement: +0.0529   95% CI [0.0411, 0.0641]   p = 0.0000
-> ESTABLISHED


### 5.6 How much headroom is left?

The k-NN overlap analysis put the Bayes error of the 9-family sub-problem at **21–38%** on these 44
features (Cover–Hart bracket from a measured 1-NN error of 0.376), i.e. an accuracy ceiling of
0.62–0.79. This model's Stage-B accuracy on true attack rows sits **inside that bracket, near its
favourable end**.

The remaining error is therefore largely a property of the **feature representation**, not of the
classifier: attack families overlap in CICFlowMeter space, and no decision rule operating on these
columns can separate them.

In [20]:
att_mask = y_test_arr != 0
sb_acc = (y_pred[att_mask] == y_test_arr[att_mask]).mean()
print("Stage-B accuracy on the %d true attack rows : %.4f" % (att_mask.sum(), sb_acc))
print("measured Bayes-accuracy bracket (k-NN)      : [0.625, 0.787]")
print("-> %s" % ("inside the bracket" if 0.625 <= sb_acc <= 0.787 else "outside the bracket -- re-check"))
print("\nbest local (k-NN) classifier on the same problem: macro-F1 0.446")
print("this ensemble, Stage B                          : macro-F1 %.4f" % best)
print("-> the model recovers ~%.2f macro-F1 beyond local neighbourhood structure" % (best - 0.446))

Stage-B accuracy on the 16234 true attack rows : 0.7331
measured Bayes-accuracy bracket (k-NN)      : [0.625, 0.787]
-> inside the bracket

best local (k-NN) classifier on the same problem: macro-F1 0.446
this ensemble, Stage B                          : macro-F1 0.6209
-> the model recovers ~0.17 macro-F1 beyond local neighbourhood structure


# 6. Model Improvement — *Fine-tune & Optimize*

The design in Stage 3 is the surviving configuration from a wider search. What was tried and
**rejected** is as much a result as what was kept, and each rejection is supported by a measurement.

| Lever | Result | Why it failed |
|---|---|---|
| `class_weight='balanced'` | 12th of 14 schemes, **below no weighting at all** | full inverse frequency demands a 128× weight ratio; the useful band is 3–11× |
| Full SMOTE | macro-F1 0.581 → 0.509 | inflates the rarest classes ~950×, interpolating noise from a handful of real points |
| Focal (difficulty-aware) weighting | γ=2 collapsed the model to 0.350 | assumes hard samples are informative; here they are ambiguous, so capacity flows into noise |
| Dynamic ensemble selection (per-instance kNN competence) | +0.004 on selection, −0.001 on test | weights collapsed to 0.333 ± 0.02 — a static ensemble in disguise |
| Algorithm-diverse pool (XGB/RF/ET at fixed α) | no gain | models disagreed on only 13% of rows; XGB dominated 1.5:1 when they did |
| α extended to 1.5 / 2.0 | 10-member pool scored *below* the 7-member | more disagreement, but from degradation rather than complementarity |
| Stacking meta-learners | 0.918 in-sample, 0.593 on test | 63 free parameters with no constraint; the linear per-class matrix has the same count and generalises |
| Per-class decision thresholds | every gain inverted on test | algebraically **redundant** with the weight matrix (verified: identical predictions on 100.0000% of rows) |
| Soft routing / gate-threshold tuning | +0.000 on test | the gate is already optimally calibrated at 0.5 |

**The pattern.** Every gain in this project came from **weighting** or from **more data** — never from
model capacity, resampling, or decision-rule adjustments. Four independent levers failed in the same
direction, which is what motivated the overlap analysis that ultimately explained the plateau.

# 7. Prediction — *Deployment & Inference*

The deployed path: raw CICFlowMeter rows in, family label and confidence out. Seven tree models plus a
7×9 matrix multiply — the members are independent and parallelise, and unlike the rejected dynamic
ensemble there is no reference set to keep in memory.

In [21]:
def predict_flows(raw: pd.DataFrame, gate_threshold: float = 0.5) -> pd.DataFrame:
    """Classify raw CICFlowMeter flows.

    raw : DataFrame with (at least) the original 76 feature columns.
    Returns label, name, attack probability and family confidence per flow.
    """
    Xq = raw[FEATURES]                                  # same 44 columns, same order
    pa = GATE.predict_proba(Xq)[:, 1]
    S  = fuse([m.predict_proba(Xq) for m in POOL], W)
    Pb = S / np.maximum(S.sum(1, keepdims=True), 1e-12)
    fam = S.argmax(1)

    lab = np.where(pa >= gate_threshold,
                   np.vectorize(idx_to_fam.get)(fam), 0)
    return pd.DataFrame({
        "label":            lab,
        "prediction":       [LABELS[v] for v in lab],
        "p_attack":         pa.round(4),
        "family_confidence": np.where(pa >= gate_threshold, Pb.max(1).round(4), np.nan),
    }, index=raw.index)

# demonstration on held-out rows, one per class where available
demo_pos = [np.where(y_test_arr == c)[0][0] for c in range(10) if (y_test_arr == c).any()]
demo = predict_flows(X_test.iloc[demo_pos])
demo.insert(0, "true", [LABELS[v] for v in y_test_arr[demo_pos]])
print(demo.to_string())
print("\ninference on %d flows:" % len(X_test))
t0 = time.time(); _ = predict_flows(X_test); dt = time.time() - t0
print("  %.1fs total -> %.0f flows/second" % (dt, len(X_test) / dt))

                  true  label      prediction  p_attack  family_confidence
37316           Benign      0          Benign    0.0000                NaN
184022        Analysis      1        Analysis    0.9644             0.9362
207211        Backdoor      7  Reconnaissance    0.9965             0.3448
18508              DoS      4        Exploits    0.9998             0.9395
225641        Exploits      5         Fuzzers    0.9740             0.4917
206280         Fuzzers      5         Fuzzers    0.9587             0.4048
173634         Generic      6         Generic    0.9939             0.9314
18646   Reconnaissance      7  Reconnaissance    0.9989             0.9992
286012       Shellcode      5         Fuzzers    0.9920             0.4142
298269           Worms      5         Fuzzers    0.9131             0.6006

inference on 61235 flows:
  2.6s total -> 23124 flows/second


## Summary

**Final model:** two-stage hierarchical detection — an XGBoost gate followed by a seven-member
α-diverse ensemble combined by a per-class weight matrix, all under tempered class weighting
(α = 0.25) and cross-fitted so the combiner never sees in-sample probabilities.

**Result:** macro-F1 ≈ **0.630 [0.611, 0.649]**, an improvement of **+0.050 [0.039, 0.061]**
(p < 0.001) over the flat `balanced` baseline.

**What the thesis can claim:**

1. Hierarchical detection **combined with tempered class weighting** significantly improves macro-F1 on
   CIC-UNSW-NB15-2024. The two are one mechanism — the hierarchy alone is not an improvement.
2. Moderate rebalancing beats full rebalancing, for **both** weighting and resampling — a single
   principle replacing two separate observations, with the useful weight-ratio band measured at 3–11×.
3. Ensemble members should be chosen for **complementarity, not individual quality**; the diversity
   that matters here runs along the class-weighting axis, not the algorithm axis.
4. The residual error is a property of the **feature representation**: attack families overlap in
   CICFlowMeter space (overlap predicts confusion at ρ = 0.83), with a measured Bayes-error bracket of
   21–38%.

**What it must not claim:** that the two-stage structure helps on its own; that improvements below
~±0.012 (paired) are real; or that rare-class point estimates are precise — Worms spans ±0.13.

**Outstanding:** SHAP per-class explainability, best targeted at the overlapping family pairs
(Exploits↔Reconnaissance, DoS↔Exploits, Fuzzers↔Shellcode) rather than at rare classes generically.

**Future work** follows directly from the overlap finding: richer features — payload inspection,
host and temporal context, session-level aggregation — rather than further classifiers over these 44
flow statistics.